In [1]:
import os
#import argparse
#parser=argparse.ArgumentParser(description='trim video into short piece')
#parser.add_argument('-i','--input_path',help='path to the video as a string: eg. C:/Users/../, apostrophe not shown',type=str,default='Z:/DATA/experiment_trackball_Optomotor')
#parser.add_argument('-n','--exp_name',help='the stimulus name of the experiment: eg. xxx, apostrophe not shown',type=str,default='gratings')
#parser.add_argument('-d','--exp_date',help='the date of the experiment',type=list,default=['220712','220714'])
#args= parser.parse_args()
#def create_data_list(input_path, exp_name, exp_date):
data_list=[]
input_path = 'Z:/DATA/experiment_trackball_Optomotor'
dataset=os.listdir(input_path)
exp_date=['220813','220811']
exp_name='sweeploom'
for locust in dataset:
    locust_directory=input_path + '/' + locust
    file_list = os.listdir(locust_directory)
    for thisDate in file_list:
        if thisDate in exp_date:
            experiment_directory=locust_directory + '/' + thisDate
            experiment_list=os.listdir(experiment_directory)
            for thisExperiment in experiment_list:
                if thisExperiment in exp_name:
                    data_list.append(experiment_directory + '/' + exp_name)
#    return data_list

#if __name__ == '__main__':
#    create_data_list(args.input_path,args.exp_name,args.exp_date)

In [8]:
import os, fnmatch
import pandas as pd
import numpy as np
import re
def generate_curated_database(myList = [], *args):
    csv_pattern='trial_*.csv'
    dat_pattern='*.dat'
    overwrite_file=False
    save_file=True
    track_ball_radius=4
    frame_rate=144
    for thisDir in myList:
        # list up the files in the dir
        file = os.listdir(thisDir)
        # look for files that matches the pattern
        csv_file= fnmatch.filter(file, csv_pattern)  
        dat_file= fnmatch.filter(file, dat_pattern)
        csv_file_directory= os.path.join(thisDir ,csv_file[0])
        dat_file_directory= os.path.join(thisDir ,dat_file[0])
        stim_info = pd.read_csv(csv_file_directory)
        stim_info=stim_info.reset_index()
        stim_variable=[]
        stimulus_timestamp=[]
        if exp_name == 'gratings':
            for row in range(0, len(stim_info)):
            #delete the bracket in current speed and save it as integer in new list
                stim_variable.append(int(stim_info.loc[row, 'Frame'].replace('(', '')))
            # combine the last two values into the timestamp (were divided into 2 columns cause german computer
            # used comma as decimal seperator and save timestamp as float in new file
                if isinstance(stim_info.loc[row, 'Value'], float):
                    timestamp = stim_info.loc[row, 'Timestamp']
                else:
                    timestamp = str(stim_info.loc[row, 'Timestamp']) + '.' + stim_info.loc[row, 'Value']
                stimulus_timestamp.append(float(timestamp.replace(')', '')))
            stim_variable=np.array(stim_variable).reshape((len(stim_info), -1))
            
        elif exp_name == 'sweeploom':
            for row in range(0, len(stim_info)):
                stim_variable.append(int(stim_info.iloc[row,2].split("=",1)[1]))
                stim_variable.append(int(stim_info.iloc[row,3].split("=",1)[1]))
                stim_variable.append(int(stim_info.iloc[row,4].split("=",1)[1]))
                stim_variable.append(int(re.split(r'=|}',stim_info.iloc[row,5])[1]))
                if isinstance(stim_info.loc[row, 'Value'], float):
                    timestamp = stim_info.loc[row, 'Timestamp']
                else:
                    timestamp = str(stim_info.loc[row, 'Timestamp']) + '.' + stim_info.loc[row, 'Value']
                stimulus_timestamp.append(float(timestamp.replace(')', '')))
            stim_variable=np.array(stim_variable).reshape((len(stim_info), -1))
            #some bugs in the bonsai workflow. Needs to remove the first timestamp and the last row of stim variable here
            stimulus_timestamp.pop(0)
            stim_variable=np.delete(stim_variable, -1, axis=0)
        elif exp_name == 'dots':
            continue ##need to add something here
        else:
            warnings.warn('stimulus paradigm can not be recognised')
        

        
        ## load dat file
        ## fictrac title is here 'frame counter', 'delta rotation vector cam x', 'delta rotation vector cam y', \
        ##'delta rotation vector cam z', 'delta rotation error score', 'delta rotation vector lab x', \
        ##'delta rotation vector lab y', 'delta rotation vector lab z', 'absolute rotation vector cam x',\
        ##'absolute rotation vector cam y', 'absolute rotation vector cam z', \
        ##'absolute rotation vector lab x', 'absolute rotation vector lab y', \
        ##'absolute rotation vector lab z', 'intergrated x position','intergrated y position', \
        ##'intergrated animal heading', 'animal movement direction', 'animal movement speed', \
        ##'intergrated forward motion','intergrated side motion', 'timestamp', 'sequence counter','delta timestamp', 'alt. timestamp']

        ## load dat file
        raw_data = pd.read_table(dat_file_directory, sep="\s+")
        
        ## drop some column
        raw_data.drop(raw_data.columns[[0,1,2,3,4,8,9,10,16,17,18,19,20,21,22,23]], axis=1, inplace=True)
        raw_data.columns=['delta rotation vector lab x', 'delta rotation vector lab y', 'delta rotation vector lab z',                       'absolute rotation vector lab x', 'absolute rotation vector lab y', 'absolute rotation vector lab z',                       'intergrated x position','intergrated y position', 'timestamp']
        ## cleaning , and change the data from object to float
        raw_data.iloc[:, :]=raw_data.iloc[:, :].replace({',': ''}, regex= True)
        for ind in range(0, raw_data.shape[1]):
            raw_data.iloc[:, ind]=pd.to_numeric(raw_data.iloc[:, ind]).astype('float32')
        
        ## adjust the unit of the x, y position
        raw_data.iloc[:, 6]=raw_data.iloc[:, 6]*track_ball_radius
        raw_data.iloc[:, 7]=raw_data.iloc[:, 7]*track_ball_radius
        ## adjust the unit of the z vector based on the target frame rate of fictrac
        raw_data.iloc[:, 2]=raw_data.iloc[:, 2]*frame_rate

        # apply filter to the raw trace
        #curr_data.loc[:, ['x_pos']] = scipy.signal.savgol_filter(curr_data.loc[:, ['x_pos']], 99, 3, axis=0)
        #curr_data.loc[:, ['y_pos']] = scipy.signal.savgol_filter(curr_data.loc[:, ['y_pos']], 99, 3, axis=0)
        
        # compare the timestamp between event file and dat file
        data_timestamp=raw_data.loc[:, 'timestamp']
        stimulus_timestamp_b=np.tile(stimulus_timestamp,(len(data_timestamp),1))
        data_timestamp_b=np.tile(data_timestamp,(len(stimulus_timestamp),1))
        timestamp_diff= data_timestamp_b > np.transpose(stimulus_timestamp_b)
        stim_onset= np.where(np.roll(timestamp_diff,1,axis=1)!=timestamp_diff)[1][1::2]
        
        ### add new column here to store event info
        raw_data['stim_onset']=False
        raw_data.loc[stim_onset, 'stim_onset']=True
        raw_data['stim_variable']=object
        for ind in range(0, len(stimulus_timestamp)):
            ## use at not loc as the data input is object not int
            raw_data.at[stim_onset[ind], 'stim_variable']=stim_variable[ind,:]
        
        ### save the curated_database
        if save_file==True:
            database_name='database_curated.pickle'
            database_directory= os.path.join(thisDir,database_name)
            if (overwrite_file ==False) and (os.path.isfile == True):
                continue
            else:
                raw_data.to_pickle(database_directory)

In [9]:
generate_curated_database(data_list)

In [ ]:
import os, fnmatch
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
def load_curated_database(myList = [], *args):
    database_pattern='*.pickle'
    stim_duration = 25 # in sec
    grey_duration = 5
    frame_rate=144
    analysis_window=frame_rate*(stim_duration+grey_duration)
    for thisDir in myList:
        file = os.listdir(thisDir)
        file_name= fnmatch.filter(file, database_pattern)
        database= os.path.join(file_dir,file_name[0])
        df=pd.read_pickle(database)
        #curr_data.loc[:, ['intergrated y position']] = savgol_filter(df.loc[:, ['intergrated x position']], 99, 3, axis=0)
        #curr_data.loc[:, ['intergrated y position']] = savgol_filter(df.loc[:, ['intergrated y position']], 99, 3, axis=0)
        stim_onset=np.where(df.loc[:,'stim_onset']==True)
        grey_onset=stim_onset-np.ones(np.shape(stim_onset),dtype=int)*grey_duration*frame_rate
        stim_offset=stim_onset+np.ones(np.shape(stim_onset),dtype=int)*stim_duration*frame_rate
        data_of_interest=np.zeros([grey_onset.shape[1],analysis_window],dtype='float32')
        for ind in range(0,grey_onset.shape[1]):
            data_of_interest[ind,:]=df.loc[grey_onset[0,ind]+1:stim_offset[0,ind],'delta rotation vector lab z']